In [ ]:
import pika
import os
import json

def get_rabbitmq_connection():
    """Build RabbitMQ connection parameters from environment variables"""
    user = os.environ.get('user', 'celery')
    password = os.environ.get('password', 'celery')
    host = os.environ.get('celery_host', '192.168.1.110')
    port = int(os.environ.get('celery_port', '31672'))
    vhost = os.environ.get('celery_vhost', 'celery')
    
    credentials = pika.PlainCredentials(user, password)
    return pika.ConnectionParameters(
        host=host,
        port=port,
        virtual_host=vhost,
        credentials=credentials
    )

def download_all_database_operation_tasks():
    """Download all messages from the database_operation queue without consuming them"""
    try:
        params = get_rabbitmq_connection()
        connection = pika.BlockingConnection(params)
        channel = connection.channel()
        
        queue_name = 'database_operation'
        tasks = []
        
        print(f"Connecting to queue: {queue_name}")
        
        # Get queue info to see how many messages are waiting
        method = channel.queue_declare(queue=queue_name, passive=True)
        message_count = method.method.message_count
        
        print(f"Messages in queue: {message_count}")
        print("-" * 80)
        
        # Retrieve all messages without acknowledging (so they stay in the queue)
        for i in range(message_count):
            method, properties, body = channel.basic_get(queue=queue_name, auto_ack=False)
            
            if method:
                try:
                    # Decode the message body
                    message_data = json.loads(body.decode('utf-8')) if body else {}
                    tasks.append({
                        'message_number': i + 1,
                        'delivery_tag': method.delivery_tag,
                        'content_type': properties.content_type,
                        'content_encoding': properties.content_encoding,
                        'headers': properties.headers,
                        'body': message_data
                    })
                    print(f"Message {i+1}: {message_data}")
                except json.JSONDecodeError:
                    # If not JSON, just return raw body
                    tasks.append({
                        'message_number': i + 1,
                        'delivery_tag': method.delivery_tag,
                        'body': body.decode('utf-8', errors='ignore')
                    })
                    print(f"Message {i+1} (raw): {body.decode('utf-8', errors='ignore')}")
        
        connection.close()
        
        print("-" * 80)
        print(f"✓ Downloaded {len(tasks)} tasks from database_operation queue")
        
        return {
            'total_messages': message_count,
            'tasks': tasks
        }
        
    except Exception as e:
        print(f"✗ Failed to download tasks: {e}")
        raise

# Execute the function
result = download_all_database_operation_tasks()
print(f"\nResult: {result['total_messages']} messages retrieved")